# Amdahl's law on a real forward pass
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week1/Amdahl_Forward_Pass_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 1, Module 1: Why accelerators**

Amdahl's law says that the part of a workload you do not speed up sets a ceiling on the total speedup. If 5 percent of a fixed workload cannot be accelerated, the maximum speedup is 20x, no matter how fast the rest becomes. Accelerators are built to make matrix multiplication fast, so for an LLM the question is how much of the time is matrix multiplication and how much is everything else.

In this notebook you profile two forward passes of Qwen 2.5 0.5B on the GPU: a prefill pass over a 512-token prompt and a single decode step. You split the GPU time into matrix multiplies and other work, and compute the ceiling Amdahl's law gives for each pass. Then you compare GPU time with wall-clock time to see how much of a decode step is spent outside the GPU.

**What you will do**
1. Check the GPU and load the model in FP16.
2. Write Amdahl's law as a function and reproduce the 20x example from the lecture.
3. Profile a prefill pass and a decode step.
4. Group the GPU time into matrix multiplies, attention and everything else.
5. Compute the speedup ceiling for each pass, first for GPU time and then for wall-clock time.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells in order. The notebook takes about three minutes, most of it downloading the model.

## 1. Check the environment
This cell prints the versions and the GPU. Record them: the split between matrix multiplies and other work depends on the GPU, the PyTorch version and the kernels it selects.

In [ ]:
import torch
import transformers

print("PyTorch:     ", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Set Runtime > Change runtime type > T4 GPU, then run this cell again.")

## 2. Amdahl's law
For a workload where a fraction `f` of the time can be sped up by a factor `s`, the new time is `(1 - f) + f / s` of the old time. The speedup is the inverse.

The lecture example: if 95 percent of the work is accelerated and the other 5 percent is not, the speedup can never exceed 1 / 0.05 = 20x. The table below shows how quickly the curve flattens as `s` grows.

In [ ]:
def amdahl(f, s):
    """Speedup when a fraction f of the time runs s times faster."""
    return 1.0 / ((1.0 - f) + f / s)

print(f"{'accelerated fraction':>22} {'s = 10':>8} {'s = 100':>8} {'s = 1000':>9} {'s = inf':>8}")
for f in (0.50, 0.80, 0.90, 0.95, 0.99):
    row = [amdahl(f, s) for s in (10, 100, 1000)]
    print(f"{f:>22.2f} {row[0]:>7.1f}x {row[1]:>7.1f}x {row[2]:>8.1f}x {1 / (1 - f):>7.1f}x")

**Reading the output**
- The last column is the ceiling: `1 / (1 - f)`. At `f = 0.95` it is 20x, the example from the lecture.
- Going from `s = 100` to `s = 1000` barely changes the result once `f` is below 0.99. The part you did not accelerate dominates.
- The rest of this notebook measures `f` for a real model instead of assuming it.

## 3. Load the model
Qwen 2.5 0.5B Instruct is a decoder-only transformer with 24 layers. It fits easily on a T4 in FP16. The download is about 1 GB.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype).to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params / 1e6:.0f} M, dtype {next(model.parameters()).dtype}, device {device}")

## 4. A helper that splits GPU time by kind of work
The PyTorch profiler records every ATen operator on the CPU and every kernel on the GPU, and it attributes each kernel's GPU time to the operator that launched it. The helper below adds up that **self GPU time** per operator and puts each operator in one of three groups:

| Group | ATen operators |
|---|---|
| matrix multiply | `aten::mm`, `aten::addmm`, `aten::bmm`, `aten::baddbmm` |
| attention kernel | the fused scaled-dot-product attention kernels |
| everything else | layer norm, activation functions, elementwise adds and multiplies, rotary embeddings, copies, indexing |

Only CPU-side operator rows are counted. The raw kernel rows are skipped because their time is already included in the operator that launched them.

In [ ]:
import statistics
import time
from torch.profiler import profile, ProfilerActivity

MATMUL_OPS = {"aten::mm", "aten::addmm", "aten::bmm", "aten::baddbmm"}
ATTENTION_PREFIXES = ("aten::_scaled_dot_product", "aten::_efficient_attention",
                      "aten::_flash_attention", "aten::scaled_dot_product")


def self_device_ms(event):
    # Newer PyTorch calls this field self_device_time_total; older versions use self_cuda_time_total.
    us = getattr(event, "self_device_time_total", None)
    if us is None:
        us = getattr(event, "self_cuda_time_total", 0)
    return us / 1000.0


def split_gpu_time(prof):
    """Return milliseconds of GPU time per group, from one profiler run."""
    groups = {"matrix multiply": 0.0, "attention kernel": 0.0, "everything else": 0.0}
    for e in prof.key_averages():
        if e.device_type != torch.autograd.DeviceType.CPU:
            continue  # raw kernel rows; their time is already counted under an operator
        ms = self_device_ms(e)
        if ms <= 0:
            continue
        if e.key in MATMUL_OPS:
            groups["matrix multiply"] += ms
        elif e.key.startswith(ATTENTION_PREFIXES):
            groups["attention kernel"] += ms
        else:
            groups["everything else"] += ms
    return groups


def wall_ms(fn, repeats=10):
    """Median wall-clock time of fn() in milliseconds, with the GPU synchronized."""
    times = []
    for _ in range(repeats):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if device == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return statistics.median(times)


activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device == "cuda" else [])
print("Profiler activities:", activities)

## 5. Profile a prefill pass
Prefill processes the whole prompt in one forward pass. Here the prompt is 512 tokens at batch size 1. Every weight matrix is multiplied by a 512-row activation matrix, so the matrix multiplies are large.

The cell runs three warm-up passes first. The first passes include one-time costs such as kernel selection, and they would distort the measurement.

In [ ]:
PROMPT_TOKENS = 512
prompt = torch.randint(0, tokenizer.vocab_size, (1, PROMPT_TOKENS), device=device)


def prefill():
    with torch.inference_mode():
        return model(input_ids=prompt, use_cache=True)


for _ in range(3):
    prefill()  # warm-up

with profile(activities=activities) as prof_prefill:
    prefill()
    if device == "cuda":
        torch.cuda.synchronize()

prefill_groups = split_gpu_time(prof_prefill)
prefill_wall = wall_ms(prefill)
gpu_total = sum(prefill_groups.values())
print(f"Prefill, {PROMPT_TOKENS} tokens")
for name, ms in prefill_groups.items():
    print(f"  {name:17s}: {ms:7.2f} ms  ({ms / gpu_total:5.1%} of GPU time)")
print(f"  {'GPU total':17s}: {gpu_total:7.2f} ms")
print(f"  {'wall clock':17s}: {prefill_wall:7.2f} ms")

**Reading the output**
- Matrix multiplies take about two thirds of the GPU time on a T4. This is the work accelerators are designed for: large matrix multiplies with a lot of reuse of each weight.
- "Everything else" is about a quarter of the GPU time. These operators move data and do little arithmetic, so they do not get faster when the matrix units get faster. They alone cap the speedup from a faster matrix unit at about 3x.
- GPU time and wall-clock time are close. The GPU is busy for most of the pass.

## 6. Profile a decode step
A decode step processes one new token. It reuses the KV cache from the prefill pass, so it does not recompute the prompt. Every weight matrix is now multiplied by a single row, which is much less work per byte of weights read.

The cell profiles five consecutive decode steps and divides by five. Each step appends one entry to the KV cache, which is too small a change to affect the result.

In [ ]:
DECODE_STEPS = 5
out = prefill()
cache = out.past_key_values
next_token = out.logits[:, -1:].argmax(-1)


def decode_steps(n=DECODE_STEPS):
    global cache, next_token
    with torch.inference_mode():
        for _ in range(n):
            step = model(input_ids=next_token, past_key_values=cache, use_cache=True)
            cache = step.past_key_values
            next_token = step.logits[:, -1:].argmax(-1)


decode_steps(3)  # warm-up

with profile(activities=activities) as prof_decode:
    decode_steps()
    if device == "cuda":
        torch.cuda.synchronize()

decode_groups = {k: v / DECODE_STEPS for k, v in split_gpu_time(prof_decode).items()}
decode_wall = wall_ms(decode_steps, repeats=5) / DECODE_STEPS
gpu_total = sum(decode_groups.values())
print("Decode, one step")
for name, ms in decode_groups.items():
    print(f"  {name:17s}: {ms:7.3f} ms  ({ms / gpu_total:5.1%} of GPU time)")
print(f"  {'GPU total':17s}: {gpu_total:7.3f} ms")
print(f"  {'wall clock':17s}: {decode_wall:7.3f} ms")
print(f"  GPU busy for {gpu_total / decode_wall:.0%} of the step")

**Reading the output**
- The matrix-multiply share of GPU time is about the same as in prefill. Each matrix multiply now has one row, so it does far less arithmetic, but it still has to read every weight, and that reading dominates its time.
- Wall-clock time is several times larger than GPU time: the GPU is busy for roughly a quarter of the step. For the rest, the GPU waits while Python and the PyTorch dispatcher prepare and launch the next of several hundred small kernels.
- This is the lecture's second Amdahl lesson: if most of the time is outside the accelerator, making the accelerator faster changes little.

## 7. Speedup ceilings
Now apply Amdahl's law to the measurements. Two versions of `f` are computed for each pass:

- **GPU view:** `f` is the matrix-multiply share of GPU time. This is the best case if only the GPU kernels counted.
- **Wall-clock view:** `f` is the matrix-multiply time divided by the wall-clock time. Time outside the GPU is part of the serial fraction.

The ceiling is the speedup if matrix multiplies took no time at all. The second column uses a 10x faster matrix unit.

In [ ]:
def ceilings(groups, wall):
    mm = groups["matrix multiply"]
    f_gpu = mm / sum(groups.values())
    f_wall = mm / wall
    return f_gpu, f_wall


print(f"{'pass':8} {'view':11} {'f':>6} {'10x matmul':>11} {'ceiling':>9}")
for name, groups, wall in (("prefill", prefill_groups, prefill_wall), ("decode", decode_groups, decode_wall)):
    f_gpu, f_wall = ceilings(groups, wall)
    for view, f in (("GPU", f_gpu), ("wall clock", min(f_wall, 0.999))):
        print(f"{name:8} {view:11} {f:6.2f} {amdahl(f, 10):10.2f}x {1 / (1 - f):8.2f}x")

In [ ]:
import matplotlib.pyplot as plt

# Stacked bars, one per pass, in grayscale. The white segment is time outside the GPU.
labels = ["prefill (512 tokens)", "decode (1 token)"]
rows = [prefill_groups, decode_groups]
walls = [prefill_wall, decode_wall]
shades = {"matrix multiply": "0.15", "attention kernel": "0.45", "everything else": "0.75"}

fig, ax = plt.subplots(figsize=(7, 3.2))
for i, (groups, wall) in enumerate(zip(rows, walls)):
    left = 0.0
    for name, ms in groups.items():
        share = ms / wall
        ax.barh(i, share, left=left, color=shades[name], edgecolor="black", label=name if i == 0 else None)
        left += share
    ax.barh(i, max(0.0, 1 - left), left=left, color="white", edgecolor="black",
            label="outside the GPU" if i == 0 else None)
ax.set_yticks(range(len(labels)), labels)
ax.set_xlim(0, 1)
ax.set_xlabel("share of wall-clock time")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=4, frameon=False, fontsize=8)
ax.set_title("Where the time goes in one forward pass")
plt.tight_layout()
plt.show()

**Reading the output**
- For prefill, the two views are close, and the ceiling, about 3x, is set by the non-matmul GPU work.
- For decode, the wall-clock ceiling is close to 1x. Even infinitely fast matrix multiplies leave most of the step unchanged, because the step is dominated by time outside the GPU.
- The fix for decode is not a faster matrix unit. It is fewer kernel launches and less host work, which Week 3 (launch overhead, CUDA Graphs) and Week 6 (the overhead term) cover.

## Try this
1. Change `PROMPT_TOKENS` to 64 and run sections 5 and 7 again. The matrix multiplies get smaller, and the matmul share of prefill drops.
2. Run section 6 with a batch of 8 sequences instead of 1. Matrix multiplies do more work per weight read, and the GPU busy fraction rises.
3. Look at the full operator table with `print(prof_decode.key_averages().table(sort_by="self_cuda_time_total", row_limit=15))` and find the largest "everything else" operators.

## Summary
- Amdahl's law: the ceiling is `1 / (1 - f)`, set by the part you do not accelerate.
- In prefill, matrix multiplies dominate GPU time, so a faster matrix unit helps.
- In a decode step, much of the wall-clock time is outside the GPU, so the ceiling from faster matrix units is low.
- Profile the full path before choosing what to optimize.

Lecture: Week 1, Module 1, slides 10 to 16 (Amdahl's law, Amdahl in AI, arithmetic intensity, roofline).